In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import PROCESSED, RESULTS, savefig
from pypsa_helpers import solve_scenario, require_optimal, total_co2, total_system_cost

import pypsa
import pandas as pd
import matplotlib.pyplot as plt

Scenario 1: No CO2 Limit

Loads the network built by `06_building_the_PyPSA_model_draft.ipynb` and solves it
unconstrained - no `GlobalConstraint` on CO2 is added, so the model is free to run the
existing CCGT fleet as much as it wants.

In [ ]:
n = pypsa.Network(str(PROCESSED / "pypsa_network_base.nc"))
cost_scale = n.meta["cost_scale"]

n = solve_scenario(n, co2_limit=None)

In [ ]:
print("status:", n.meta["status"], "| termination condition:", n.meta["termination_condition"])

# Refuse to report numbers from a solve that didn't actually finish: HiGHS's objective
# reads back as 0.0 whenever termination_condition != "optimal" (crossover not run), even
# though p_nom_opt/dispatch are already populated with a not-yet-feasible iterate - that
# looks like "zero cost" or "the CO2 cap didn't work" but is really just an unfinished solve.
require_optimal(n)

total_cost_eur = total_system_cost(n, cost_scale=cost_scale)
total_co2_t = total_co2(n)

print(f"total annual system cost: {total_cost_eur:,.0f} EUR/yr")
print(f"total CO2 emissions: {total_co2_t:,.0f} t/yr")

In [ ]:
n.export_to_netcdf(str(PROCESSED / "pypsa_results_no_co2_limit.nc"))

Built Capacity by Technology

In [ ]:
capacity_by_carrier = n.generators.groupby("carrier").p_nom_opt.sum().add(
    n.storage_units.groupby("carrier").p_nom_opt.sum(), fill_value=0
)

fig, ax = plt.subplots(figsize=(8, 5))
capacity_by_carrier.plot.bar(ax=ax)
ax.set_ylabel("Optimal capacity (MW)")
ax.set_title("No CO2 Limit: Built Capacity by Technology")

savefig(fig, "pypsa", "no_co2_limit_capacity.png")
plt.show()